
# Editorial GPU Validation — Evidence-Driven Movie Essay

Run this notebook on a **GPU runtime** (T4/A100) to produce a 60–120s
cinematic movie analysis from a **real movie you legally own**, using the
**same code the orchestrator runs** (`python src/main.py run`).

### Pipeline under test (the NEW editorial architecture)
```
movie -> Whisper -> PySceneDetect scenes -> Qwen director -> thesis
      -> evidence retrieval -> editorial_plan.json
      -> evidence-aligned script -> editorial timeline
      -> excerpt clips -> Kokoro TTS -> FFmpeg render -> QC
```
The AI "editor" plans the cut (hook / thesis / evidence segments / close),
retrieves specific on-screen moments as evidence, writes narration that
corresponds to that evidence, produces short cinematic captions, mixes the
audio, and renders a playable MP4.

### Strict modes are ON (no silent fallbacks)
- `REQUIRE_REAL_LLM=true` — real Qwen director on CUDA; mock/deterministic director is refused.
- `REQUIRE_REAL_TTS=true` — real Kokoro/Chatterbox/Qwen3-TTS; mock audio is refused.
- `EDITORIAL_MODE=true` — the editorial pipeline (not the plain montage).
- TTS runs on **CUDA only**.

### Movie is never committed
Supply a **Google Drive share link** (`MOVIE_URL`, downloaded with `gdown` — no
Drive mount needed), a path on the mounted Drive, or upload a file. Optionally
trim to the first N seconds for a cheap first run.

### Artifacts to inspect after the run
`director_plan.json`, `editorial_plan.json`, `script.json`,
`timeline/editorial_timeline.json`, `audio/tts_meta.json`,
`provider_manifest.json`, `reports/qc_report.json`, `renders/final_render.mp4`.



### Cell 1 — Runtime & system packages
Installs FFmpeg, a consistent CUDA PyTorch, Transformers (Qwen), Whisper and
PySceneDetect. Set `REPO_URL` / `BRANCH` to your fork/branch if needed, and
supply the movie via `MOVIE_URL` (Drive share link) or `MOVIE_PATH`.


In [ ]:

# @title 1) Setup: system + repo + base deps
import os, sys, subprocess

REPO_URL = "https://github.com/asdfhgds/automovies.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
MOVIE_URL = ""  # @param {type:"string"} Google Drive share link (e.g. https://drive.google.com/file/d/xxx/view)
MOVIE_PATH = ""  # @param {type:"string"} Drive/local path (used if MOVIE_URL is empty)

def sh(cmd, **kw):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    sys.stdout.write(p.stdout or "")
    if p.returncode != 0:
        sys.stdout.write(p.stderr or "")
        raise SystemExit(f"command failed ({p.returncode}): {cmd}\n--- tail ---\n{(p.stderr or '')[-4000:]}")

# Always work from an absolute, deterministic location so re-running this cell
# never nests extra copies of the repo.
ROOT = "/content"
REPO_DIR = os.path.join(ROOT, "automovies")
os.chdir(ROOT)
if not os.path.isdir(os.path.join(REPO_DIR, "src")):
    if os.path.isdir(REPO_DIR):
        sh(f"rm -rf {REPO_DIR}")
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}")
else:
    # Already cloned: fetch the latest fixes/scripts without wiping anything.
    sh(f"git -C {REPO_DIR} fetch origin {BRANCH}")
    sh(f"git -C {REPO_DIR} reset --hard origin/{BRANCH}")
os.chdir(REPO_DIR)
sh("bash scripts/colab_setup.sh")

# Remember the movie source for the next cells.
if MOVIE_PATH:
    open("/content/movie_path.txt", "w").write(MOVIE_PATH)
if MOVIE_URL:
    open("/content/movie_url.txt", "w").write(MOVIE_URL)
print("Setup complete. Repo:", os.getcwd())



### Cell 2 — Google Drive + TTS deps
Mount Drive so a movie can be supplied without committing it (optional if
`MOVIE_URL` is set), then install the open-source TTS stack (Kokoro required;
Chatterbox / Qwen3-TTS optional). Restart-free: this step restores a
Transformers version that still provides `Qwen3ForCausalLM`.


In [ ]:

# @title 2) Mount Drive + install TTS deps
from google.colab import drive
drive.mount("/content/drive")

sh("bash scripts/colab_tts_setup.sh")

def _read(p):
    try:
        return open(p).read().strip()
    except FileNotFoundError:
        return ""

MOVIE_PATH = _read("/content/movie_path.txt")
MOVIE_URL = _read("/content/movie_url.txt")
print("MOVIE_URL =", repr(MOVIE_URL))
print("MOVIE_PATH =", repr(MOVIE_PATH))



### Cell 3 — Get + validate the movie
Priority: `MOVIE_URL` (Drive share link) → `MOVIE_PATH` (Drive/local path) →
upload. The file id is extracted and normalized to the `uc?id=` form gdown
handles reliably. The result is verified to actually be a video. A link that
needs permissions, or a non-video file, fails loudly here instead of
mid-pipeline. Optionally `TRIM_TO_SEC` cuts the movie to its first N seconds so
a long/expensive title can be tested cheaply first (trim to e.g. 180–300s so
the evidence selector has enough footage).


In [ ]:

# @title 3) Get + validate movie file
import os, re, subprocess
from IPython.display import display, HTML

TRIM_TO_SEC = 0  # @param {type:"number"} Prefer a short section first? Enter seconds to trim the movie to its first N seconds (0 = full movie).

def _read(p):
    try:
        return open(p).read().strip()
    except FileNotFoundError:
        return ""

MOVIE_PATH = _read("/content/movie_path.txt")
MOVIE_URL = _read("/content/movie_url.txt")

# Tolerate a URL accidentally pasted into the MOVIE_PATH field
if MOVIE_PATH.startswith("http"):
    MOVIE_URL = MOVIE_URL or MOVIE_PATH
    MOVIE_PATH = ""

# Normalize any Drive URL to the uc?id= form gdown parses natively.
if MOVIE_URL:
    m = re.search(r"(?:file/d/|id=)([a-zA-Z0-9_-]{10,})", MOVIE_URL)
    if m:
        fid = m.group(1)
        MOVIE_URL = f"https://drive.google.com/uc?id={fid}&export=download"
        print("Using Drive file id:", fid)

if MOVIE_URL and not MOVIE_PATH:
    try:
        import gdown
    except ImportError:
        sh("pip install -q gdown")
        import gdown
    print("Downloading movie from Google Drive link:", MOVIE_URL)
    saved = gdown.download(MOVIE_URL, output="/content/movie_download", quiet=False)
    assert saved and os.path.exists(saved), "gdown failed to download the movie"
    MOVIE_PATH = saved
    size = os.path.getsize(MOVIE_PATH)
    print(f"Downloaded {size/1e6:.1f} MB to {MOVIE_PATH}")
    assert size > 1024 * 1024, (
        f"Downloaded file is only {size} bytes - this is the error page, not the "
        "movie. Make sure the Drive link points to a real video file shared as "
        "'Anyone with the link'."
    )
    open("/content/movie_path.txt", "w").write(MOVIE_PATH)

if not MOVIE_PATH:
    from google.colab import files
    print("Uploading a movie from this machine (or set MOVIE_URL / Drive path above):")
    up = files.upload()
    MOVIE_PATH = list(up.keys())[0]
    open("/content/movie_path.txt", "w").write(MOVIE_PATH)

MOVIE_PATH = os.path.abspath(os.path.expanduser(MOVIE_PATH))
print("Movie:", MOVIE_PATH, "exists:", os.path.exists(MOVIE_PATH))
assert os.path.exists(MOVIE_PATH), "Movie file not found"

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration,size",
     "-show_entries", "stream=codec_type,codec_name,width,height",
     "-of", "default=noprint_wrappers=1", MOVIE_PATH],
    capture_output=True, text=True)
print(probe.stdout or probe.stderr)
assert "codec_type=video" in (probe.stdout or ""), (
    "The file at MOVIE_PATH is not a valid video. If using MOVIE_URL, make sure "
    "it is a real video file shared as 'Anyone with the link'."
)

if TRIM_TO_SEC and int(TRIM_TO_SEC) > 0:
    trimmed = "/content/movie_trimmed.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", MOVIE_PATH,
         "-t", str(int(TRIM_TO_SEC)), "-c", "copy", trimmed], check=True)
    MOVIE_PATH = trimmed
    open("/content/movie_path.txt", "w").write(MOVIE_PATH)
    print(f"Trimmed to first {int(TRIM_TO_SEC)}s -> {MOVIE_PATH}")

print("OK: valid video file.")



### Cell 4 — Environment + doctor
Enables the `colab-gpu` profile, both strict modes, and the **editorial**
pipeline (`EDITORIAL_MODE=true`). `TTS_PROVIDER=kokoro` is the default; switch
to `chatterbox` / `qwen3_tts` if you installed them. `TTS_DEVICE=cuda`
guarantees real TTS never falls back to CPU. `EDITORIAL_TARGET_SEC` steers the
essay length (60–120s recommended here). `EDITORIAL_CREATIVE_TASK` optionally
adds a creative brief for the director.


In [ ]:

# @title 4) Env vars + doctor (editorial pipeline)
import os
os.environ["STUDIO_PROFILE"] = "colab-gpu"
os.environ["REQUIRE_REAL_LLM"] = "true"
os.environ["REQUIRE_REAL_TTS"] = "true"
os.environ["DIRECTOR_PROVIDER"] = "qwen"
os.environ["DIRECTOR_MODEL"] = "Qwen/Qwen3-4B-Instruct-2507"
os.environ["TTS_DEVICE"] = "cuda"
os.environ["TTS_PROVIDER"] = "kokoro"  # kokoro | chatterbox | qwen3_tts
os.environ["TTS_VOICE"] = "am_adam"
os.environ["EDITORIAL_MODE"] = "true"        # the AI editor builds the cut
os.environ["EDITORIAL_TARGET_SEC"] = "90"    # 60-120s essay target
os.environ["EDITORIAL_CREATIVE_TASK"] = ""   # optional creative brief
os.environ["RUN_TTS_BENCHMARK"] = "false"    # optional: set to "true"
os.environ["BURN_SUBTITLES"] = "true"

import subprocess as _sp
dr = _sp.run(["python", "src/main.py", "doctor"], capture_output=True, text=True)
out = (dr.stdout or dr.stderr or "").strip()
print(out[-3000:] if len(out) > 3000 else out)



### Cell 5 — Initialize the project
Registers the real movie (path only, not the file) under `data/<project-id>`.


In [ ]:

# @title 5) init project
import subprocess, re
from pathlib import Path

MOVIE_PATH = Path(open("/content/movie_path.txt").read().strip()).expanduser().resolve()
assert MOVIE_PATH.exists(), "no movie was registered - run Cell 3 first"
out = subprocess.run(
    ["python", "src/main.py", "init",
     "--title", "Real Movie Video Essay",
     "--source", str(MOVIE_PATH)],
    capture_output=True, text=True)
print(out.stdout or out.stderr)
m = re.search(r"project ([0-9a-f-]{36})", out.stdout or "")
PROJECT_ID = m.group(1) if m else None
print("PROJECT_ID =", PROJECT_ID)
assert PROJECT_ID, "failed to init project"
open("/content/project_id.txt", "w").write(PROJECT_ID)



### Cell 6 — Run the full EDITORIAL pipeline (the long one)
Real Qwen director on CUDA, evidence retrieval, editorial plan + script,
excerpt extraction, real Kokoro TTS on CUDA, FFmpeg render with film/music
ducking, loudness normalization, a true-peak limiter, and burned short
captions. This is the same entry point the CLI uses, so nothing in the
notebook is special to it. Allow up to 2h for a full-length movie
(transcription + full-frame scene detection + LLM + TTS + render).


In [ ]:

# @title 6) run EDITORIAL pipeline (real LLM + real TTS + editorial render)
import subprocess, time

PROJECT_ID = open("/content/project_id.txt").read().strip()
t0 = time.time()
out = subprocess.run(
    ["python", "src/main.py", "run", "--project-id", PROJECT_ID],
    timeout=7200)
dt = time.time() - t0
print(f"--- pipeline wall time: {dt:.1f}s ---")
print("(exit code)", out.returncode)
if out.returncode != 0:
    raise SystemExit("pipeline failed (see logs above)")



### Cell 7 — QC + ffprobe validation + manifest
Runs the QC critic (artifact existence, real-TTS flag, no clipping via
`volumedetect`, render probe), prints the provider manifest, and previews the
render in-line.


In [ ]:

# @title 7) QC + validation + manifest + preview
import json, subprocess, sys
from pathlib import Path

sys.path.insert(0, "src")

PROJECT_ID = open("/content/project_id.txt").read().strip()
proj = Path("data") / PROJECT_ID

from qc.critic import run_qc
report = run_qc(proj)
print(json.dumps(report, indent=2))

manifest = json.loads((proj / "provider_manifest.json").read_text())
print("\n=== PROVIDER MANIFEST (excerpt) ===")
for k in ("strict_mode", "profile", "editorial_mode", "editorial_plan_built",
          "editorial_timeline_built", "director_real_generation", "director_model",
          "director_device", "tts_provider", "tts_model", "tts_device", "tts_real",
          "transcription_real", "pipeline_total_seconds"):
    print(f"  {k}: {manifest.get(k)}")

render = proj / "renders" / "final_render.mp4"
probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-show_entries", "stream=codec_type,codec_name,width,height",
     "-of", "json", str(render)], capture_output=True, text=True)
print("\n=== RENDER (ffprobe) ===")
print(probe.stdout)
from IPython.display import Video
display(Video(str(render)))



### Cell 8 — Inspect the creative + editorial decisions, then download
Shows the thesis, hook, evidence selected per segment, and the number of
excerpt clips in the timeline, then downloads `final_render.mp4` to your
machine.


In [ ]:

# @title 8) Inspect decisions + download the render
import json
from pathlib import Path

PROJECT_ID = open("/content/project_id.txt").read().strip()
proj = Path("data") / PROJECT_ID

plan = json.loads((proj / "editorial_plan.json").read_text())
print("TITLE:", plan.get("title"))
print("THESIS:", plan.get("thesis"))
print("HOOK:", (plan.get("hook") or {}).get("text"))
print()
for seg in plan.get("segments", []):
    print(f"- [{seg['id']}] {seg['purpose']}")
    for ev in seg.get("evidence", []):
        print(f"    evidence {ev['scene_id']} {ev['start_sec']}-{ev['end_sec']}s: {ev['reason'][:90]}")
    print("    narration:", seg.get("narration", {}).get("text", "")[:110])

tl = json.loads((proj / "timeline" / "editorial_timeline.json").read_text())
print("\nExcerpt clips:", sum(len(seg.get("video", [])) for seg in tl.get("segments", [])))
print("Timeline total (s):", tl.get("total_duration_sec"))

from google.colab import files
print("\nDownloading final_render.mp4 ...")
files.download(str(proj / "renders" / "final_render.mp4"))



### How to evaluate the edited short (do this before the next milestone)
A valid MP4 is **not** success. Watch the video and score:

1. **Idea** — Is the thesis interesting / original?
2. **Movie understanding** — Does it understand what the selected scenes mean?
3. **Script** — Analysis, not a generic summary?
4. **Evidence** — Does narration match what is actually shown?
5. **Editing** — Intentional cuts, pacing, meaning created by the edit?
6. **TTS** — Natural voice / real performance, or robotic reading?
7. **Subtitles** — Short and readable? Synced? Highlighting?
8. **Audio** — Narration clear; film/dialogue ducked appropriately?
9. **Overall** — Would you publish this?

Log what you observe. The largest visible weakness decides the next milestone
(Movie Understanding, Semantic Evidence Retrieval, Editorial Director,
TTS Performance, Subtitle System, Audio Design, or Visual Generation)
**it should not be decided by a predetermined feature list**.
